# LAMBDA DIARIA

In [ ]:
import requests
import boto3
from datetime import datetime, timedelta
import time
import io
import csv

s3 = boto3.client('s3')

API_KEY = "******************"
BASE_URL = "https://api.rawg.io/api/games"
BUCKET_NAME = "proyecto-final-rawg"

def min_ratings_by_year(year: int) -> int:
    if year < 2000:
        return 5
    elif year < 2010:
        return 15
    else:
        return 40

def lambda_handler(event, context):
    try:
        today = datetime.utcnow()
        yesterday = today - timedelta(days=1)

        date_from = yesterday.strftime("%Y-%m-%d")
        date_to = today.strftime("%Y-%m-%d")

        print(f"Buscando juegos entre {date_from} y {date_to}")

        page = 1
        games = []
        seen_ids = set()

        while True:
            params = {
                "key": API_KEY,
                "page": page,
                "page_size": 40,
                "dates": f"{date_from},{date_to}",
                "ordering": "-metacritic",
                "metacritic": "1,100"
            }

            response = requests.get(BASE_URL, params=params, timeout=30)

            if response.status_code == 404:
                break

            response.raise_for_status()
            data = response.json()

            results = data.get("results", [])
            if not results:
                break

            for game in results:
                try:
                    game_id = game.get("id")
                    if game_id in seen_ids:
                        continue

                    name = game.get("name")
                    slug = game.get("slug")
                    released = game.get("released")
                    rating = game.get("rating")
                    metacritic = game.get("metacritic")
                    ratings_count = game.get("ratings_count")
                    reviews_text_count = game.get("reviews_text_count")
                    added = game.get("added")
                    playtime = game.get("playtime")

                    platforms_data = game.get("platforms", [])
                    genres_data = game.get("genres", [])
                    developers_data = game.get("developers", [])
                    publishers_data = game.get("publishers", [])
                    tags = game.get("tags", [])
                    suggestions_count = game.get("suggestions_count")

                    if not name:
                        continue
                    if metacritic is None:
                        continue
                    if not released:
                        continue
                    if not genres_data:
                        continue

                    year = int(released[:4])
                    min_ratings = min_ratings_by_year(year)

                    if ratings_count is None or ratings_count < min_ratings:
                        continue

                    num_platforms = len(platforms_data) if platforms_data else 0

                    multiplayer = True if any(
                        isinstance(tag, dict)
                        and tag.get("language") == "eng"
                        and str(tag.get("name", "")).lower() == "multiplayer"
                        for tag in tags
                    ) else False

                    genres = ", ".join(
                        g["name"] for g in genres_data
                        if isinstance(g, dict) and g.get("name")
                    )

                    platforms = ", ".join(
                        p["platform"]["name"] for p in platforms_data
                        if isinstance(p, dict)
                        and p.get("platform")
                        and p["platform"].get("name")
                    )

                    developers = ", ".join(
                        d["name"] for d in developers_data
                        if isinstance(d, dict) and d.get("name")
                    )

                    publishers = ", ".join(
                        p["name"] for p in publishers_data
                        if isinstance(p, dict) and p.get("name")
                    )

                    if not genres:
                        continue

                    row = [
                        name,
                        slug,
                        released,
                        rating,
                        ratings_count,
                        reviews_text_count,
                        metacritic,
                        added,
                        suggestions_count,
                        genres,
                        platforms,
                        developers,
                        publishers,
                        playtime,
                        year,
                        num_platforms,
                        multiplayer
                    ]

                    games.append(row)
                    seen_ids.add(game_id)

                except Exception as e:
                    print(f"Error procesando juego: {e}")
                    continue

            print(f"Página {page} | Total válidos: {len(games)}")
            page += 1
            time.sleep(1)

        if not games:
            return {"statusCode": 200, "body": "No hay juegos válidos"}

        csv_buffer = io.StringIO()
        writer = csv.writer(csv_buffer)

        writer.writerow([
            "name",
            "slug",
            "released",
            "rating",
            "ratings_count",
            "reviews_text_count",
            "metacritic",
            "added",
            "suggestions_count",
            "genres",
            "platforms",
            "developers",
            "publishers",
            "playtime",
            "year",
            "num_platforms",
            "multiplayer"
        ])

        writer.writerows(games)

        file_name = f"rawg_{date_from}.csv"

        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=file_name,
            Body=csv_buffer.getvalue(),
            ContentType="text/csv"
        )

        print(f"Archivo subido: {file_name}")

        return {
            "statusCode": 200,
            "body": f"{len(games)} juegos válidos subidos a S3"
        }

    except Exception as e:
        print(f"ERROR: {str(e)}")
        return {
            "statusCode": 500,
            "body": str(e)
        }